In [4]:
# 02_risk_classification.ipynb
# =============================================================================
# Personalized Health Intervention Pathways for Comorbid Policyholders:
# Loss Ratio Management via LLM Guardrails and Counterfactual Explanations
# -----------------------------------------------------------------------------
# Notebook 02 — XGBoost Risk Classification (Age × Sex Stratified Models)
# Input  : ../results/tables/knhanes_comorbid_2020_2024.csv
# Output : ../results/tables/model_{group}.pkl
#          ../results/tables/agent_config.pkl
#          ../results/tables/df_final.pkl
#          ../results/tables/model_performance.csv
# =============================================================================

# %%
# ## 0. Import Libraries

import joblib
import warnings
import numpy as np
import pandas as pd
import optuna
import xgboost as xgb

from sklearn.metrics import (
    classification_report, f1_score, roc_auc_score
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# %%
# ## 1. Load Preprocessed Dataset

df_total = pd.read_csv('../results/tables/knhanes_comorbid_2020_2024.csv')
print(f"Dataset shape: {df_total.shape}")

# %%
# ## 2. Stratification Configuration

AGEGROUP_CONFIG = {
    "MiddleAged_Male":   {"age_min": 40, "age_max": 59, "sex_code": 1.0},
    "MiddleAged_Female": {"age_min": 40, "age_max": 59, "sex_code": 2.0},
    "Older_Male":        {"age_min": 60, "age_max": 99, "sex_code": 1.0},
    "Older_Female":      {"age_min": 60, "age_max": 99, "sex_code": 2.0},
}

# %%
# ## 3. Feature Definition

NUM_FEATURES = [
    'BMI', 'WaistCirc', 'Weight',
    'Energy_kcal', 'Carb_g', 'Sugar_g', 'Sodium_mg',
    'Fat_g', 'SaturatedFat_g', 'Fiber_g', 'Potassium_mg', 'Protein_g',
]
CAT_FEATURES = [
    'ObesityStatus', 'WeightChangeStatus', 'WeightLossAmount', 'WeightGainAmount',
    'DrinkingFrequency', 'DrinkingAmount', 'SmokingStatus',
    'VigorousActivity_Work', 'VigorousActivity_Leisure',
    'ModerateActivity_Work', 'WalkingActivity', 'AerobicActivityRate',
    'BreakfastFrequency', 'StressLevel', 'StressAwarenessRate',
    'PersonalIncomeQuartile', 'HouseholdIncomeQuartile',
    'EducationLevel', 'HealthScreeningStatus',
]
X_FEATURES = NUM_FEATURES + CAT_FEATURES
TARGET_COL = 'RiskClass'

# Fixed features: not varied in DiCE search (socioeconomic factors)
FIXED_FEATURES = [
    'StressLevel', 'StressAwarenessRate',
    'PersonalIncomeQuartile', 'HouseholdIncomeQuartile',
    'EducationLevel', 'HealthScreeningStatus',
]
VARY_FEATURES = [f for f in X_FEATURES if f not in FIXED_FEATURES]

# %%
# ## 4. Build Analysis Dataset

required_cols = X_FEATURES + [TARGET_COL, 'Sex', 'AgeGroup']
df_final = df_total[required_cols].copy()
for col in df_final.columns:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce').fillna(0)
df_final = df_final.astype(float)

print(f"df_final shape: {df_final.shape}")
print(f"Any object columns: {(df_final.dtypes == 'object').any()}")

# %%
# ## 5. XGBoost Model Training — 4 Stratified Models

def train_model(age_group: float, gender_code: float, group_name: str):
    """
    Train an XGBoost multi-class risk model for one stratified group.

    Parameters
    ----------
    age_group   : 0.0 = MiddleAged (40–59), 1.0 = Older (60+)
    gender_code : 1.0 = Male, 2.0 = Female
    group_name  : label used for logging and file saving

    Returns
    -------
    best_model       : fitted XGBClassifier
    best_params      : dict of optimal hyperparameters
    X_val, y_val     : held-out validation set
    """
    print(f"\n{'='*20} [{group_name}] Model Training {'='*20}")

    df_g = df_final[
        (df_final['AgeGroup'] == age_group) &
        (df_final['Sex']      == gender_code)
    ].copy()

    print(f"  Sample size: {len(df_g)}")
    print(f"  Class distribution:\n{df_g[TARGET_COL].value_counts().sort_index()}")

    if len(df_g) < 100:
        print(f"  WARNING: insufficient sample ({len(df_g)}). Skipping.")
        return None, None, None, None

    X = df_g[X_FEATURES]
    y = df_g[TARGET_COL].astype(int)

    # Stratified 80/20 split
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    sw = compute_sample_weight('balanced', y=y_train)

    # Optuna hyperparameter search (weighted F1)
    def objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
            'max_depth':        trial.suggest_int('max_depth', 3, 7),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'subsample':        trial.suggest_float('subsample', 0.7, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
            'objective':        'multi:softprob',
            'num_class':        4,
            'tree_method':      'hist',
            'random_state':     42,
        }
        mdl = xgb.XGBClassifier(**params)
        mdl.fit(X_train, y_train, sample_weight=sw)
        return f1_score(y_val, mdl.predict(X_val), average='weighted')

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=10)

    best_model = xgb.XGBClassifier(
        **study.best_params,
        objective='multi:softprob',
        num_class=4,
        tree_method='hist',
        random_state=42,
    )
    best_model.fit(X_train, y_train, sample_weight=sw)

    print(f"\n[{group_name}] Classification Report:")
    print(classification_report(y_val, best_model.predict(X_val)))

    return best_model, study.best_params, X_val, y_val


# Train all four stratified models
models, params_dict, val_sets = {}, {}, {}

for group_name, cfg in AGEGROUP_CONFIG.items():
    age_grp = 0.0 if cfg['age_min'] < 60 else 1.0
    mdl, prm, X_val, y_val = train_model(
        age_group   = age_grp,
        gender_code = cfg['sex_code'],
        group_name  = group_name,
    )
    models[group_name]      = mdl
    params_dict[group_name] = prm
    val_sets[group_name]    = (X_val, y_val)

# %%
# ## 6. Model Performance Evaluation

print("\n" + "="*60)
print("Group-level Performance Metrics")
print("="*60)

perf_records = []

for group_name, cfg in AGEGROUP_CONFIG.items():
    mdl = models.get(group_name)
    if mdl is None:
        continue

    age_grp = 0.0 if cfg['age_min'] < 60 else 1.0
    df_g = df_final[
        (df_final['AgeGroup'] == age_grp) &
        (df_final['Sex']      == cfg['sex_code'])
    ].copy()

    X = df_g[X_FEATURES]
    y = df_g[TARGET_COL].astype(int)

    # Macro OvR AUC
    y_bin  = label_binarize(y, classes=[0, 1, 2, 3])
    y_prob = mdl.predict_proba(X)
    try:
        auc_macro = roc_auc_score(y_bin, y_prob,
                                  multi_class='ovr', average='macro')
    except Exception:
        auc_macro = float('nan')

    # Class 3 OvR AUC
    y_bin3 = (y == 3).astype(int)
    try:
        auc_class3 = roc_auc_score(y_bin3, y_prob[:, 3])
    except Exception:
        auc_class3 = float('nan')

    # 5-Fold CV weighted F1
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        mdl, X, y, cv=skf, scoring='f1_weighted', n_jobs=-1
    )

    # Validation set metrics
    X_val, y_val = val_sets[group_name]
    report = classification_report(
        y_val, mdl.predict(X_val), output_dict=True
    )
    acc = report['accuracy']
    p3  = report.get('3', {}).get('precision', float('nan'))
    r3  = report.get('3', {}).get('recall',    float('nan'))
    f3  = report.get('3', {}).get('f1-score',  float('nan'))

    print(f"\n[{group_name}]")
    print(f"  Overall Accuracy  : {acc:.4f}")
    print(f"  Class 3 Precision : {p3:.4f}")
    print(f"  Class 3 Recall    : {r3:.4f}")
    print(f"  Class 3 F1        : {f3:.4f}")
    print(f"  AUC (macro OvR)   : {auc_macro:.4f}")
    print(f"  AUC (Class 3 OvR) : {auc_class3:.4f}")
    print(f"  5-Fold CV F1      : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    perf_records.append({
        'Group':       group_name,
        'N':           len(df_g),
        'Accuracy':    round(acc,        4),
        'Class3_P':    round(p3,         4),
        'Class3_R':    round(r3,         4),
        'Class3_F1':   round(f3,         4),
        'AUC_macro':   round(auc_macro,  4),
        'AUC_class3':  round(auc_class3, 4),
        'CV_F1_mean':  round(cv_scores.mean(), 4),
        'CV_F1_std':   round(cv_scores.std(),  4),
    })

perf_df = pd.DataFrame(perf_records)
perf_df.to_csv('../results/tables/model_performance.csv',
               index=False, encoding='utf-8-sig')
print("\n>>> Saved: ../results/tables/model_performance.csv")
print(perf_df.to_string(index=False))

# %%
# ## 7. Save Models and Configuration

agent_config = {
    'X_features':      X_FEATURES,
    'num_features':    NUM_FEATURES,
    'cat_features':    CAT_FEATURES,
    'vary_features':   VARY_FEATURES,
    'fixed_features':  FIXED_FEATURES,
    'target_col':      TARGET_COL,
    'agegroup_config': AGEGROUP_CONFIG,
    'best_params':     params_dict,
}

for group_name, mdl in models.items():
    if mdl is not None:
        joblib.dump(mdl, f'../results/tables/model_{group_name}.pkl')

joblib.dump(agent_config, '../results/tables/agent_config.pkl')
joblib.dump(df_final,     '../results/tables/df_final.pkl')
print(">>> Models and configuration saved.")

Dataset shape: (6339, 39)
df_final shape: (6339, 34)
Any object columns: False

==================== [MiddleAged_Male] Model Training ====================
  Sample size: 1088
  Class distribution:
RiskClass
0.0    615
1.0    196
2.0    106
3.0    171
Name: count, dtype: int64

[MiddleAged_Male] Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.87      0.78       123
           1       0.41      0.30      0.35        40
           2       0.12      0.05      0.07        21
           3       0.52      0.44      0.48        34

    accuracy                           0.62       218
   macro avg       0.44      0.41      0.42       218
weighted avg       0.57      0.62      0.58       218


==================== [MiddleAged_Female] Model Training ====================
  Sample size: 2272
  Class distribution:
RiskClass
0.0    1785
1.0     242
2.0     143
3.0     102
Name: count, dtype: int64

[MiddleAged_Female] Classification Report